In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [3]:
# Define file paths
from pathlib import Path

try:
    PROJECT_DIR = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_DIR = Path.cwd().parent

INPUT_FILE = PROJECT_DIR / "data" / "raw" / "all_stocks_historical.csv"
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"

# Ensure the processed folder exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



In [4]:
# Load the original dataset

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}\n"
        "Complete Step 2 and download the stock data first."
    )

df = pd.read_csv(INPUT_FILE)

required_columns = [
    "Date", "Open", "High", "Low",
    "Close", "Volume", "Ticker"
]

missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("=" * 60)
print("STOCK MARKET DATA QUALITY CHECK")
print("=" * 60)

print(f"Original rows: {len(df):,}")
print(f"Original columns: {len(df.columns)}")


STOCK MARKET DATA QUALITY CHECK
Original rows: 3,762
Original columns: 7


In [5]:
# Standardize data types

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

df["Ticker"] = (
    df["Ticker"]
    .astype("string")
    .str.strip()
    .str.upper()
)

numeric_columns = ["Open", "High", "Low", "Close", "Volume"]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Treat infinite numeric values as missing
df[numeric_columns] = df[numeric_columns].replace(
    [np.inf, -np.inf], np.nan
)


In [6]:
# Identify data-quality problems

df["_missing_date"] = df["Date"].isna()
df["_missing_ticker"] = df["Ticker"].isna() | df["Ticker"].eq("")
df["_missing_price"] = df[
    ["Open", "High", "Low", "Close"]
].isna().any(axis=1)

df["_missing_volume"] = df["Volume"].isna()

df["_nonpositive_price"] = (
    df[["Open", "High", "Low", "Close"]] <= 0
).any(axis=1)

df["_negative_volume"] = df["Volume"] < 0

df["_invalid_ohlc"] = (
    (df["High"] < df[["Open", "Close", "Low"]].max(axis=1))
    | (df["Low"] > df[["Open", "Close", "High"]].min(axis=1))
)

# Duplicate ticker-date observations
df["_duplicate_ticker_date"] = df.duplicated(
    subset=["Ticker", "Date"], keep="first"
)

# Build a quality report before cleaning
quality_rows = []

for ticker, group in df.groupby("Ticker", dropna=False):
    quality_rows.append({
        "Ticker": str(ticker),
        "Rows_before_cleaning": len(group),
        "Missing_dates": int(group["_missing_date"].sum()),
        "Missing_prices": int(group["_missing_price"].sum()),
        "Missing_volume": int(group["_missing_volume"].sum()),
        "Nonpositive_prices": int(group["_nonpositive_price"].sum()),
        "Negative_volume_rows": int(group["_negative_volume"].sum()),
        "Invalid_OHLC_rows": int(group["_invalid_ohlc"].sum()),
        "Duplicate_ticker_date_rows": int(
            group["_duplicate_ticker_date"].sum()
        ),
    })

quality_report = pd.DataFrame(quality_rows)

print("\nDATA QUALITY REPORT")
print(quality_report.to_string(index=False))



DATA QUALITY REPORT
Ticker  Rows_before_cleaning  Missing_dates  Missing_prices  Missing_volume  Nonpositive_prices  Negative_volume_rows  Invalid_OHLC_rows  Duplicate_ticker_date_rows
  AAPL                  1254              0               0               0                   0                     0                  0                           0
  MSFT                  1254              0               0               0                   0                     0                  0                           0
  NVDA                  1254              0               0               0                   0                     0                  0                           0


In [7]:
# Remove unusable or invalid observations

rows_before = len(df)

invalid_rows = (
    df["_missing_date"]
    | df["_missing_ticker"]
    | df["_missing_price"]
    | df["_nonpositive_price"]
    | df["_negative_volume"]
    | df["_invalid_ohlc"]
)

# Keep the first observation for each ticker-date pair
duplicate_rows = df.duplicated(
    subset=["Ticker", "Date"], keep="first"
)

clean = df.loc[~invalid_rows & ~duplicate_rows].copy()

# Remove temporary quality-check columns
temporary_columns = [
    col for col in clean.columns if col.startswith("_")
]
clean = clean.drop(columns=temporary_columns)

# Sort for consistent analysis
clean = clean.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

rows_removed = rows_before - len(clean)


In [8]:
# Final validation

assert clean["Date"].notna().all()
assert clean["Ticker"].notna().all()
assert not clean.duplicated(["Ticker", "Date"]).any()
assert (clean[["Open", "High", "Low", "Close"]] > 0).all().all()

assert (
    clean["High"]
    >= clean[["Open", "Close", "Low"]].max(axis=1)
).all()

assert (
    clean["Low"]
    <= clean[["Open", "Close", "High"]].min(axis=1)
).all()

# Missing volume is reported, not silently invented
remaining_missing_volume = int(clean["Volume"].isna().sum())


In [10]:
from pathlib import Path

# Re-verify project path definitions  
try:
    PROJECT_DIR = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_DIR = Path.cwd().parent

OUTPUT_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_FILE = OUTPUT_DIR / "clean_stocks_data.csv"
QUALITY_FILE = OUTPUT_DIR / "data_quality_report.csv"

# Check for missing summary counters 
if 'rows_before' not in locals():
    # Fallback to keep script running if notebook state was partially reset
    rows_before = len(clean) if 'clean' in locals() else 0
    rows_removed = 0

if 'remaining_missing_volume' not in locals():
    remaining_missing_volume = clean['Volume'].isna().sum() if 'clean' in locals() else 0


# Save clean data and quality report

clean.to_csv(CLEAN_FILE, index=False)
quality_report.to_csv(QUALITY_FILE, index=False)

print("\n" + "=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)

print(f"Rows before cleaning: {rows_before:,}")
print(f"Rows removed: {rows_removed:,}")
print(f"Rows after cleaning: {len(clean):,}")
print(f"Stocks retained: {clean['Ticker'].nunique()}")
print(f"Duplicate ticker-date pairs remaining: "
      f"{clean.duplicated(['Ticker', 'Date']).sum()}")
print(f"Missing volume values remaining: {remaining_missing_volume}")

print("\nRows by ticker:")
print(clean.groupby("Ticker").size().to_string())

print(f"\nClean dataset saved to:\n{CLEAN_FILE}")
print(f"\nQuality report saved to:\n{QUALITY_FILE}")

print("\nData cleaning and validation completed.")



CLEANING SUMMARY
Rows before cleaning: 3,762
Rows removed: 0
Rows after cleaning: 3,762
Stocks retained: 3
Duplicate ticker-date pairs remaining: 0
Missing volume values remaining: 0

Rows by ticker:
Ticker
AAPL    1254
MSFT    1254
NVDA    1254

Clean dataset saved to:
/Users/golden_horde/Desktop/FA_projects/stock-market-analysis/data/processed/clean_stocks_data.csv

Quality report saved to:
/Users/golden_horde/Desktop/FA_projects/stock-market-analysis/data/processed/data_quality_report.csv

Data cleaning and validation completed.
